# PKG counterparty entity map v2 + name typer v2 — `pkg_cpty_entity_map_v2.ipynb`

**Why.** The threshold exploration showed that the v1 business flag misses most businesses: `person_other` held 82% of
counterparty entities and 50% of dollars. The examples showed the causes. Suffixes behind punctuation were missed
(`GENERAL ATLANTIC (TRA), L.P.`), as were foreign suffixes (`SCSP`), institutional words without a suffix (`CLEVELAND
CLINIC`, `OHS COLLECTION ACCOUNT`), and digit-only names (`75010501`, Treasury wires). Digit-only names were also never
merged, so one counterparty appeared as many single-transaction entities. Check payees carry an address in the name.
v1 merges also included bare first names (JOHN, DAVID).

**What v2 does.**

1. **Name typer v2.** Every raw `cpty_id` gets a `name_class` and a `type_reason` explaining which rule fired.
   Precedence: `no_name` → `placeholder` → `numeric_name` → `bank` → `business` (tokens) → `person` (positive evidence
   only) → `business` (behaviour) → `person` (P2P name shape) → **`unknown`**. Nothing defaults to person any more.
2. **First names from the data.** First tokens of short alphabetic P2P names, kept when they lead far more often than
   they trail. No external list.
3. **Entity map v2**, in a new table `pkg_cpty_entity_map_v2` (v1 untouched):
   - `ent_l1` — unchanged (6C ids folded into their 5C id).
   - `ent_l2` — businesses merged on a cleaned name key (address stripped, suffixes normalised), ignoring the FI;
     digit-only names merged on name + FI; bare first names and generic single words never merge.
   - `ent_l2_fi` — stricter: the same merges, but only within one FI.
   - `name_class` / `type_reason` / `name_clean` per id, so downstream notebooks read the class from the map.

**Reading the output.** §3 tells you whether the dollars move where they should: the v1 `person_other` dollars should
land mostly in `business` / `unknown`, and `person` should hold small amounts. §4 is the eyeball check: the largest
`person` and `unknown` ids by dollars. Edit the vocabulary lists in §0b and rerun from §3; the typed table is cached.

## §0 Setup

In [ ]:
# §0a  Session, shims, helpers
import os, json, time, warnings, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                        ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F

SHUFFLE_PARTITIONS = 2000
spark = (SparkSession.builder
         .appName("pkg_cpty_entity_map_v2")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .enableHiveSupport()
         .getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))

# ---- display: one output per cell ----
def _fmt_val(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if kind == "pct":
        v = 0.0 if abs(v) < 5e-7 else v
        return "0" if v == 0 else (f"{v:.3f}" if abs(v) < 1 else f"{v:.2f}")
    if kind == "musd":
        return f"{v:,.1f}"
    if kind in ("usd", "int"):
        return f"{v:,.0f}"
    return f"{v:,.2f}"

def fmt_df(df, fmt=None):
    out = df.copy()
    out.columns.name = None
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if not pd.api.types.is_numeric_dtype(s):
            out[c] = s.where(s.notna(), "")
            continue
        name = str(c)
        kind = fmt or ("pct" if "%" in name else "musd" if name.startswith("$M") or name.startswith("$B")
                       else "usd" if name.startswith("$")
                       else "int" if pd.api.types.is_integer_dtype(s) or bool((s.dropna() % 1 == 0).all()) else "f2")
        out[c] = [_fmt_val(v, kind) for v in s]
    return out

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 9px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#8a8984;margin:2px 0}</style>")

def note(text):
    return ("__note__", text)

def show(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        else:
            fmt = b[2] if len(b) > 2 else None
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(b[0])}</b></div>")
            parts.append(fmt_df(b[1], fmt).to_html(index=False, escape=True, border=0, classes="pkgt"))
    display(HTML(_CSS + "<div style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>" + "".join(parts) + "</div>"))

_jvm, _hconf = spark._jvm, spark._jsc.hadoopConfiguration()
def _fs(p):
    path = _jvm.org.apache.hadoop.fs.Path(p)
    return path.getFileSystem(_hconf), path
def path_done(p):
    fs, path = _fs(p.rstrip("/") + "/_SUCCESS")
    return fs.exists(path)
def path_gb(p):
    fs, path = _fs(p)
    return fs.getContentSummary(path).getLength() / 1e9 if fs.exists(path) else 0.0
def path_delete(p):
    fs, path = _fs(p)
    return bool(fs.exists(path) and fs.delete(path, True))
def db_location(db):
    for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {db}").collect():
        if str(r[0]).strip().lower() == "location":
            return str(r[1]).rstrip("/")
    raise RuntimeError(f"no Location row in DESCRIBE DATABASE {db}")
print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# §0b  Config and vocabularies (edit the lists, then rerun from §3 with FORCE_TYPE = True)
DB         = "bdahd01p_dlcdi1_cdi_tm"
T_CPTY_DIM = f"{DB}.pkg_cpty_dim"
T_MAP_V1   = f"{DB}.pkg_cpty_entity_map"
T_MAP_V2   = f"{DB}.pkg_cpty_entity_map_v2"      # written in §5
FORCE_TYPE = False          # retype even if the cached typed table exists
WRITE_MAP  = True           # §5 writes T_MAP_V2

FN_MIN_IDS     = 200        # a first name must lead at least this many P2P names …
FN_LEAD_RATIO  = 3.0        # … and lead ≥ this many times more often than it trails
KEY_MIN_LEN    = 4          # shortest cleaned business key allowed to merge
WIRE_INST_MIN  = 1_000_000  # wire-only ids with at least this many lifetime dollars are institutions
NUM_SHARE      = 0.8        # digits / alphanumerics at or above this → numeric_name
PERSON_MAX_AMT = 10_000_000 # person rules apply only below this lifetime amount; above it a non-business name is
                            # typed business / high_value (an individual moving > $10M through PNC customers is rare)
VOCAB_MIN_IDS  = 300        # learned business word: in ≥ this many legal-suffix / org-word names …
VOCAB_RATIO    = 20.0       # … and ≥ this many times more often than in P2P person names
DF_MAX         = 300        # a merge key is distinctive if it has a word (≥ 4 letters) in ≤ this many distinct keys;
                            # distinctive keys merge across FIs, generic keys (FIRST BAPTIST CHURCH) only within an FI
CONC_MIN       = 20         # … or if it averages ≥ this many ids per FI (national brands: CIGNA 43, INTUIT 769K;
                            # local look-alikes: FIRST BAPTIST CHURCH 2, HOLIDAY INN EXPRESS 3)
NO_MERGE_KEYS  = []         # keys you never want merged, e.g. ["FLEX PAYMENT ACCOUNT"]
SHORT_KEYS_OK  = ["UPS", "BP", "DHL", "HP", "IBM", "GE", "3M", "CVS", "ATT", "UBS", "AIG", "HSN", "QVC", "KFC", "UMR"]
BRAND_BEH_SHARE = 0.8       # a key that is a blocked single word (DELTA, WENDY, UNITED) or 3 letters (CDS, NDC) still
                            # merges when ≥ this share of its ids are merchant / ACH-originator descriptors and it is
                            # concentrated (≥ CONC_MIN ids per FI): card and ACH descriptors name brands, not people
# Brand aliases: regex on the cleaned key → canonical key (applied before merging; always merged across FIs)
KEY_ALIASES = {r"^(AMAZON|AMZN)\b.*": "AMAZON", r"^(MICROSOFT|MSFT)\b.*": "MICROSOFT",
               r"^(SOUTHWES\w*|SWA)\b.*": "SOUTHWEST AIRLINES", r"^(FACEBK|FACEBOOK)\b.*": "FACEBOOK"}
# Numbers that identify a separate organisation and are kept in the key (POST 123, LOCAL 456)
KEEP_NUM_AFTER = ["POST", "LOCAL", "CHAPTER", "LODGE", "TROOP", "COUNCIL", "DISTRICT", "WARD"]
TEST_NAME_RE   = r"SMOKE ?TEST|\bTEST(ER)? ?(USER|ACCOUNT|ACCT|NAME|\d+)\b"   # test records in production data

# Legal-form tokens, matched after punctuation is removed and runs of single letters are joined (L.P. → LP)
LEGAL = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LLLP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY",
         "PC", "PLLC", "PA", "PLC", "SCSP", "SCS", "SARL", "SA", "AG", "GMBH", "BV", "NV", "SPA", "SRL", "PTE", "PTY",
         "ULC", "DBA", "LTDA", "SAS", "KG", "AB", "OYJ", "NA"]
# Words that make a name an organisation even without a legal suffix
ORG = ["HOLDINGS", "HOLDING", "FUND", "FUNDS", "CAPITAL", "PARTNERS", "PARTNERSHIP", "INVESTORS", "INVESTMENT",
       "INVESTMENTS", "MANAGEMENT", "AGGREGATOR", "ACCOUNT", "COLLECTION", "COLLECTIONS", "ESCROW", "TITLE",
       "SETTLEMENT", "TRUST", "TRUSTEE", "ESTATE OF", "CLINIC", "HOSPITAL", "HEALTH", "HEALTHCARE", "MEDICAL",
       "DENTAL", "PHARMACY", "UNIVERSITY", "COLLEGE", "SCHOOL", "SCHOOLS", "ACADEMY", "CHURCH", "MINISTRIES",
       "MINISTRY", "PARISH", "DIOCESE", "TEMPLE", "MOSQUE", "ASSOC", "ASSOCIATION", "ASSN", "ASSOCIATES", "FOUNDATION",
       "SOCIETY", "INSTITUTE", "CENTER", "CENTRE", "CLUB", "LEAGUE", "COUNCIL", "COMMITTEE", "UNION", "SERVICES",
       "SERVICE", "SOLUTIONS", "GROUP", "ENTERPRISES", "ENTERPRISE", "INDUSTRIES", "INTERNATIONAL", "SYSTEMS",
       "TECHNOLOGIES", "TECHNOLOGY", "CONSTRUCTION", "CONTRACTING", "CONTRACTORS", "BUILDERS", "REALTY",
       "PROPERTIES", "PROPERTY", "APARTMENTS", "HOMEOWNERS", "HOA", "CONDO", "CONDOMINIUM", "INSURANCE", "AGENCY",
       "MORTGAGE", "LENDING", "LOAN", "LOANS", "FINANCIAL", "FINANCE", "PAYROLL", "PAYMENTS", "TREASURY",
       "DEPARTMENT", "DEPT", "COUNTY", "CITY OF", "STATE OF", "TOWN OF", "VILLAGE OF", "TOWNSHIP", "BOROUGH",
       "AUTHORITY", "DISTRICT", "COMMISSION", "COMMONWEALTH", "GOVERNMENT", "UNITED STATES", "US DEPT", "IRS",
       "UTILITIES", "UTILITY", "ELECTRIC", "ENERGY", "POWER", "WATER", "GAS", "TELECOM", "WIRELESS",
       "COMMUNICATIONS", "MOTORS", "AUTOMOTIVE", "AUTO", "RESTAURANT", "CAFE", "GRILL", "PIZZA", "MARKET", "STORE",
       "STORES", "SHOP", "SUPPLY", "SUPPLIES", "DISTRIBUTION", "DISTRIBUTORS", "WHOLESALE", "LOGISTICS",
       "TRANSPORT", "TRANSPORTATION", "TRUCKING", "FREIGHT", "EXPRESS", "FARM", "FARMS", "LAW", "LEGAL", "ATTORNEYS",
       "CPA", "CPAS", "ACCOUNTING", "CONSULTING", "CONSULTANTS", "DESIGN", "STUDIO", "STUDIOS", "MEDIA", "MARKETING",
       "PRODUCTIONS", "VENTURES", "EQUITY", "ADVISORS", "ADVISORY", "SECURITIES", "BROKERAGE", "REIT", "LEASING",
       "RENTAL", "RENTALS", "PLUMBING", "ELECTRICAL", "ROOFING", "HVAC", "LANDSCAPING", "CLEANING", "CORPORATE",
       "TOPCO", "BIDCO", "HOLDCO", "OPCO", "ACQUISITION", "MERGER", "SUB", "SPV", "NOMINEE", "OPERATING", "PRODUCTS",
       "MANUFACTURING", "PROCESSING", "PROCESSOR", "PROCESSORS", "LABS", "LABORATORIES", "PHARMACEUTICALS", "SOFTWARE", "NETWORKS", "AIRLINES", "HOTEL",
       "HOTELS", "RESORT", "CASINO", "THEATRE", "THEATER", "MUSEUM", "LIBRARY", "PROGRAM", "PROJECT", "PLAN",
       "FEDERAL", "NATIONAL", "AMERICA", "AMERICAN", "UNITED", "GENERAL", "CREDIT UNION", "REVENUE", "TREASURER",
       "CUSTOMERS", "GATEWAY", "GATEWAYS", "MERCHANT", "RECEIPT", "WIRES", "TRANSACTION", "TRANSACTIONS", "CASH CARRY",
       "AIRCRAFT", "ENGINES", "AIR LINES", "INTERACTIVE", "DISTRIBUTORS"]
# Brand words that mark a business when typing but are valid merge keys on their own
ORG_BRANDS = ["PAYMENTECH", "PAYPAL", "WORLDPAY", "FISERV"]
# Channel / product words stripped from merge keys (CENTERPOINT ENERGY OTP = CENTERPOINT ENERGY IVR)
CHANNEL = ["OTP", "IVR", "WEB", "PPD", "CCD", "WEBPAY", "AUTOPAY", "EPAY", "ONLINE", "PMT", "PMTS", "PYMT", "PAYMENT",
           "PAYMENTS", "BILLPAY", "BILL PAY", "ACH", "DEBIT", "RECURRING"]
# Never learned as business words
STOP = ["AND", "OF", "THE", "FOR", "DE", "LA", "DEL", "LOS", "LAS", "Y", "DA", "DOS", "VAN", "DER", "MC", "ST", "LE",
        "DI", "AL", "EL", "BIN", "BEN", "ABU"]
# Tokens ignored when testing a person name (titles, generational and professional suffixes)
PERSON_EXTRA = ["MR", "MRS", "MS", "MISS", "DR", "JR", "SR", "II", "III", "IV", "MD", "DO", "DDS", "DMD", "DVM", "DC",
                "OD", "RN", "NP", "PHD", "ESQ"]
# Words that can lead a P2P name but are not first names
NOT_FIRST = ["THE", "AND", "OF", "MY", "YOUR", "OUR", "NEW", "FIRST", "BEST", "SUPER", "GREAT", "AMERICAN", "NATIONAL",
             "US", "USA", "ALL", "ONE", "BIG", "LITTLE", "GOOD", "TOP", "PRO", "ROYAL", "GOLDEN", "BLUE", "RED",
             "GREEN", "BLACK", "WHITE", "NORTH", "SOUTH", "EAST", "WEST", "CENTRAL", "SAINT", "ST", "MOUNT", "HOLY",
             "GRACE", "LOVE", "PAYPAL", "VENMO", "ZELLE", "CASH"]
GENERIC_KEYS = ["PAYMENT", "PAYMENTS", "TRANSFER", "DEPOSIT", "REFUND", "CHECK", "WIRE", "ACH", "ONLINE", "MOBILE",
                "BILL", "BILLPAY", "PAYEE", "VENDOR", "CUSTOMER", "CLIENT", "OWNER", "TENANT", "LANDLORD", "USER",
                "CHECKING", "SAVINGS", "ACCT", "ACCOUNT", "MONEY MARKET"]
PLACEHOLDERS = ["BANK ACCOUNT NAME", "UNKNOWN", "UNKNOWN NAME", "NONE", "NA", "N A", "NULL", "TEST", "CUSTOMER",
                "BENEFICIARY", "PAYEE", "ACCOUNT HOLDER", "NAME", "NO NAME", "ON FILE", "CASH", "VARIOUS", "XXX",
                "USER", "CHECKING", "SAVINGS", "ACCT", "CHECKING ACCOUNT", "SAVINGS ACCOUNT", "MY ACCOUNT"]
STREET = ["ROAD", "RD", "STREET", "ST", "AVENUE", "AVE", "AV", "BOULEVARD", "BLVD", "DRIVE", "DR", "LANE", "LN",
          "HIGHWAY", "HWY", "WAY", "COURT", "CT", "PLACE", "PL", "PARKWAY", "PKWY", "CIRCLE", "CIR", "TERRACE", "TER",
          "TRAIL", "TRL", "PIKE", "ROUTE", "RT", "RTE", "SQUARE", "SQ", "SUITE", "STE", "APT", "UNIT", "FLOOR", "FL"]

TMP_ROOT = f"{db_location(DB)}/_tmp_pkg_cpty_map_v2"
P_TYPED  = f"{TMP_ROOT}/typed"
P_MAP_V2 = f"{db_location(DB)}/pkg_cpty_entity_map_v2"
print("typed cache:", P_TYPED, "| map v2 data:", P_MAP_V2)

## §1 Inputs

In [ ]:
# §1  Inputs and checks
def cols_of(t):
    return dict(spark.table(t).dtypes)

ddt, mdt = cols_of(T_CPTY_DIM), cols_of(T_MAP_V1)
def pick(dt, table, candidates, required=True):
    for c in candidates:
        if c in dt:
            return c
    assert not required, f"{table}: none of {candidates}; has {sorted(dt)}"
    return None

DIM = {"name": pick(ddt, T_CPTY_DIM, ["cpty_name", "dominant_name", "name"]),
       "fi": pick(ddt, T_CPTY_DIM, ["fi", "cpty_fin_entity_name", "dominant_fi"]),
       "cpty_type": pick(ddt, T_CPTY_DIM, ["cpty_type", "dominant_cpty_type"]),
       "categories": pick(ddt, T_CPTY_DIM, ["categories"]),
       "amount": pick(ddt, T_CPTY_DIM, ["amount"]),
       "n_txn": pick(ddt, T_CPTY_DIM, ["n_txn"]),
       "is_business": pick(ddt, T_CPTY_DIM, ["is_business"]),
       "name_is_fi": pick(ddt, T_CPTY_DIM, ["name_is_fi"]),
       "is_placeholder": pick(ddt, T_CPTY_DIM, ["is_placeholder"])}
for c in ["cpty_id", "ent_l1", "cpty_id_src"]:
    assert c in mdt, f"{T_MAP_V1}: missing {c}"
assert ddt["cpty_id"] == "string" and mdt["cpty_id"] == "string", "cpty_id must be string in both tables"

# categories may be an array or a delimited string; normalise to "1B,1C,…"
_cat = F.col(DIM["categories"])
CAT_STR = (F.concat_ws(",", _cat) if ddt[DIM["categories"]].startswith("array")
           else F.regexp_replace(_cat.cast("string"), r"[\[\]\"' ]", ""))

base = (spark.table(T_CPTY_DIM)
        .select("cpty_id",
                F.col(DIM["name"]).alias("name"), F.col(DIM["fi"]).alias("fi"),
                F.col(DIM["cpty_type"]).alias("cpty_type"), CAT_STR.alias("cats"),
                F.col(DIM["amount"]).cast("double").alias("amt"), F.col(DIM["n_txn"]).cast("long").alias("n_txn"),
                F.col(DIM["is_business"]).cast("boolean").alias("v1_biz"),
                F.col(DIM["name_is_fi"]).cast("boolean").alias("v1_fi"),
                F.col(DIM["is_placeholder"]).cast("boolean").alias("v1_ph"))
        .join(spark.table(T_MAP_V1).select("cpty_id", "cpty_id_src", "ent_l1"), "cpty_id", "inner"))

r = spark.table(T_CPTY_DIM).agg(F.count("*").alias("dim_rows"), F.sum(F.col(DIM["amount"]).cast("double")).alias("amt")).first()
m = spark.table(T_MAP_V1).agg(F.count("*").alias("map_rows")).first()
show("§1 Inputs",
     ("Tables", pd.DataFrame([{"table": T_CPTY_DIM, "rows": r["dim_rows"], "$B lifetime": r["amt"] / 1e9},
                              {"table": T_MAP_V1, "rows": m["map_rows"], "$B lifetime": np.nan}])),
     ("Dim columns used", pd.DataFrame([{"role": k, "column": v} for k, v in DIM.items()])),
     note("The map and the dim must have the same ids; §5 asserts the v2 map has exactly the v1 id set."))

## §2 Name normalisation and the first-name list

- `nm` — upper-case, punctuation to spaces, address tail removed (a house number followed within a few words by a street
  word, or a PO box), e.g. `JULIAN OBREGON AND JULIAN OBREGON 22400 COUNTY ROAD 48` → `JULIAN OBREGON AND JULIAN OBREGON`.
- `nm_c` — `nm` with runs of single letters joined (`L P` → `LP`, `N A` → `NA`), used for suffix and keyword matching.
- First names: the first token of short alphabetic P2P names (and the token after the comma in `LAST, FIRST`), kept when
  seen leading ≥ `FN_MIN_IDS` ids and ≥ `FN_LEAD_RATIO` times more often than trailing. That ratio removes surnames.
- Learned business words: tokens of names that carry a legal suffix or organisation word, kept when they appear in
  ≥ `VOCAB_MIN_IDS` such names and ≥ `VOCAB_RATIO`× more often than in P2P person names (SUBARU, FEDERAL, …).
- Token flags per id: is token 1 / token 2 a first name, does any token look like a first name, any learned business
  word. One broadcast join over the exploded tokens.

In [ ]:
# §2  Normalised names and the data-derived first-name list
def wre(words):                       # whole-word alternation
    return r"\b(" + "|".join(sorted(set(words), key=len, reverse=True)) + r")\b"

up   = F.upper(F.trim(F.col("name")))
nm0  = F.trim(F.regexp_replace(F.regexp_replace(up, r"[^A-Z0-9]+", " "), r"\s+", " "))
ADDR = (r"^(.*?[A-Z].*?)\s+(?:\d{1,6}[A-Z]?\s+(?:[A-Z0-9]+\s+){0,5}?(?:" + "|".join(STREET) + r")\b|P\s?O\s+BOX\b|BOX\s+\d).*$")
nm   = F.trim(F.regexp_replace(nm0, ADDR, "$1"))
nm_c = F.regexp_replace(nm, r"\b([A-Z]) (?=[A-Z]\b)", "$1")          # letters only: "5 L P" → "5 LP", not "5LP"
nm_p = F.trim(F.regexp_replace(F.regexp_replace(nm, wre(PERSON_EXTRA), " "), r"\s+", " "))
after_comma = F.trim(F.regexp_replace(F.regexp_replace(F.upper(F.substring_index(F.col("name"), ",", -1)),
                                                       r"[^A-Z]+", " "), r"\s+", " "))

named = (base
         .withColumn("nm", nm).withColumn("nm_c", nm_c).withColumn("nm_p", nm_p)
         .withColumn("has_comma", F.col("name").contains(","))
         .withColumn("ft1", F.element_at(F.split(F.col("nm_p"), " "), 1))
         .withColumn("ftc", F.when(F.col("has_comma"), F.element_at(F.split(after_comma, " "), 1)))
         .withColumn("n_tok", F.size(F.split(F.col("nm_p"), " ")))
         .withColumn("alpha", F.col("nm_p").rlike(r"^[A-Z]+( [A-Z]+)*$"))
         .withColumn("biz_tok", F.col("nm_c").rlike(wre(LEGAL + ORG))))

# First names from P2P names: leading vs trailing counts per token
p2p = named.where((F.col("cpty_type") == "P2P_CPTY") & F.col("alpha") & F.col("n_tok").between(2, 3) & ~F.col("biz_tok"))
toks = F.split(F.col("nm_p"), " ")
lead = F.when(F.col("has_comma") & F.col("ftc").isNotNull(), F.col("ftc")).otherwise(F.element_at(toks, 1))
trail = F.when(F.col("has_comma"), F.element_at(toks, 1)).otherwise(F.element_at(toks, -1))
fn_stats = (p2p.select(lead.alias("tok"), F.lit(1).alias("lead"), F.lit(0).alias("trail"))
            .unionByName(p2p.select(trail.alias("tok"), F.lit(0).alias("lead"), F.lit(1).alias("trail")))
            .groupBy("tok").agg(F.sum("lead").alias("lead"), F.sum("trail").alias("trail"))
            .where((F.length("tok") >= 2) & (F.col("lead") >= FN_MIN_IDS)
                   & (F.col("lead") >= FN_LEAD_RATIO * F.greatest(F.col("trail"), F.lit(1))))
            .where(~F.col("tok").isin(NOT_FIRST + LEGAL + ORG))
            .toPandas().sort_values("lead", ascending=False))
FIRST_NAMES = set(fn_stats.tok)
FN_DF = F.broadcast(spark.createDataFrame(pd.DataFrame({"tok": sorted(FIRST_NAMES)})))

# Learned business words: frequent in organisation names, rare in person names
def tok_counts(df, col, label):
    return (df.select("cpty_id", F.explode(F.array_distinct(F.split(F.col(col), " "))).alias("tok"))
              .where(F.col("tok").rlike(r"^[A-Z]{3,}$")).groupBy("tok").agg(F.count("*").alias(label)))
biz_src = named.where(F.col("biz_tok") & ~F.coalesce(F.col("v1_ph"), F.lit(False))
                      & ~F.coalesce(F.col("v1_fi"), F.lit(False)) & ~F.col("nm").isin(PLACEHOLDERS))
per_src = p2p
vocab = (tok_counts(biz_src, "nm_c", "biz_ids").join(tok_counts(per_src, "nm_p", "person_ids"), "tok", "left")
         .fillna(0, ["person_ids"])
         .where((F.col("biz_ids") >= VOCAB_MIN_IDS) & (F.col("biz_ids") >= VOCAB_RATIO * (F.col("person_ids") + 1)))
         .where(~F.col("tok").isin(STOP + LEGAL + PERSON_EXTRA))
         .toPandas())
vocab = vocab[~vocab.tok.isin(FIRST_NAMES)].sort_values("biz_ids", ascending=False)
BIZ_VOCAB = set(vocab.tok)

# Per-id token flags (first 8 tokens of the person-cleaned name)
TOKV = F.broadcast(spark.createDataFrame(pd.DataFrame(
    [(t, t in FIRST_NAMES, t in BIZ_VOCAB) for t in sorted(FIRST_NAMES | BIZ_VOCAB)], columns=["tok", "is_fn", "is_bv"])))
tok_flags = (named.select("cpty_id", F.posexplode(F.slice(F.split(F.col("nm_p"), " "), 1, 8)).alias("pos", "tok"))
                  .join(TOKV, "tok", "inner")
                  .groupBy("cpty_id")
                  .agg(F.max((F.col("is_fn") & (F.col("pos") == 0)).cast("int")).alias("fn_t1"),
                       F.max((F.col("is_fn") & (F.col("pos") == 1)).cast("int")).alias("fn_t2"),
                       F.max(F.col("is_fn").cast("int")).alias("fn_any"),
                       F.max(F.col("is_bv").cast("int")).alias("bv_any")))

top = fn_stats.head(40).assign(ratio=lambda d: d.lead / d.trail.clip(lower=1))
show(f"§2 First names derived from P2P names: {len(FIRST_NAMES):,}",
     note(f"Lead ≥ {FN_MIN_IDS:,} ids and ≥ {FN_LEAD_RATIO:g}× more often first than last. "
          "If surnames or business words show up below, raise the thresholds or add them to NOT_FIRST."),
     ("Top 40", top.rename(columns={"tok": "token", "lead": "ids leading", "trail": "ids trailing", "ratio": "lead/trail"})),
     None if len(fn_stats) <= 40 else ("40 at the bottom of the list", fn_stats.tail(40).rename(
         columns={"tok": "token", "lead": "ids leading", "trail": "ids trailing"})),
     note(f"Learned business words: {len(BIZ_VOCAB):,} (≥ {VOCAB_MIN_IDS} organisation names, ≥ {VOCAB_RATIO:g}× "
          "person names). Remove a word by adding it to STOP."),
     ("Learned business words — top 40", vocab.head(40).rename(columns={"tok": "token", "biz_ids": "org names",
                                                                         "person_ids": "person names"})),
     None if len(vocab) <= 40 else ("Learned business words — bottom 40", vocab.tail(40).rename(
         columns={"tok": "token", "biz_ids": "org names", "person_ids": "person names"})))

## §3 Typer v2

Rules in order; the first that fires sets `name_class` and `type_reason`.

| # | name_class | type_reason | Rule |
|---|---|---|---|
| 1 | no_name | `no_name` | name null or blank |
| 2 | placeholder | `placeholder` | v1 placeholder flag, or the cleaned name is in `PLACEHOLDERS` |
| 3 | numeric_name | `numeric` | digits ≥ `NUM_SHARE` of the alphanumerics |
| 4 | bank | `bank_name` / `name_is_fi` | v1 bank-like flag, or the name is the holding FI's name (`HUNTINGTON NATIONAL BA` at HUNTINGTON NATIONAL BANK) |
| 5 | business | `legal_suffix` / `org_word` / `learned_word` | a legal-form token, an organisation word, or a learned business word |
| 6 | business | `ach_originator` / `merchant` | `cpty_type` NON_PNC_ACH_ORIGINATOR / MERCHANT (people do not originate ACH) |
| 7 | person | `comma_name` / `first_name` / `last_first` / `joint_names` | below `PERSON_MAX_AMT` only: `LAST, FIRST` with a first name after the comma; 2–4 alphabetic tokens with a first name in position 1, or in position 2 (`DAVIS CAROLYN M`); `A AND B` with any first name |
| 8 | business | `wire_institution` | wire-only, ≥ `WIRE_INST_MIN` lifetime dollars |
| 9 | business | `high_value` | ≥ `PERSON_MAX_AMT` lifetime dollars with no person evidence that applies (RAYMOND JAMES, STAR OHIO, DELL) |
| 10 | person | `p2p_shape` | P2P_CPTY with 2–4 alphabetic tokens |
| 11 | unknown | `unknown` | none of the above |

In [ ]:
# §3a  Type every cpty_id (cached)
def build_typed():
    fnc = FN_DF.select(F.col("tok").alias("ftc"), F.lit(True).alias("fnc"))
    alnum = F.length(F.regexp_replace(F.col("nm"), r"[^A-Z0-9]", ""))
    digits = F.length(F.regexp_replace(F.col("nm"), r"[^0-9]", ""))
    wire_only = F.col("cats").rlike(r"^4[A-C](,4[A-C])*$")
    t = (named.join(fnc, "ftc", "left").join(tok_flags, "cpty_id", "left")
         .withColumn("fnc", F.coalesce("fnc", F.lit(False)))
         .fillna(0, ["fn_t1", "fn_t2", "fn_any", "bv_any"]))
    # the name is the holding FI's own name (prefix match on letters and digits, ≥ 6 characters)
    nc = F.regexp_replace(F.col("nm"), r"[^A-Z0-9]", "")
    fc = F.regexp_replace(F.upper(F.coalesce(F.col("fi"), F.lit(""))), r"[^A-Z0-9]", "")
    name_is_fi = ((F.length(nc) >= 6) & (F.length(fc) >= 6)
                  & ((F.expr("left(regexp_replace(upper(coalesce(fi, '')), '[^A-Z0-9]', ''), "
                             "length(regexp_replace(nm, '[^A-Z0-9]', '')))") == nc)
                     | (F.expr("left(regexp_replace(nm, '[^A-Z0-9]', ''), "
                               "length(regexp_replace(upper(coalesce(fi, '')), '[^A-Z0-9]', '')))") == fc)))
    shape = F.col("alpha") & F.col("n_tok").between(2, 4) & ~F.col("nm_p").rlike(wre(["THE", "AT", "OF", "ST"]))
    can_person = F.col("amt") < PERSON_MAX_AMT
    c = (F.when(F.col("name").isNull() | (F.col("nm") == ""), F.array(F.lit("no_name"), F.lit("no_name")))
          .when(F.coalesce(F.col("v1_ph"), F.lit(False)) | F.col("nm").isin(PLACEHOLDERS) | F.col("nm").rlike(TEST_NAME_RE),
                F.array(F.lit("placeholder"), F.lit("placeholder")))
          .when((alnum > 0) & (digits >= NUM_SHARE * alnum), F.array(F.lit("numeric_name"), F.lit("numeric")))
          .when(F.coalesce(F.col("v1_fi"), F.lit(False)), F.array(F.lit("bank"), F.lit("bank_name")))
          .when(name_is_fi, F.array(F.lit("bank"), F.lit("name_is_fi")))
          .when(F.col("nm_c").rlike(wre(LEGAL)), F.array(F.lit("business"), F.lit("legal_suffix")))
          .when(F.col("nm_c").rlike(wre(ORG + ORG_BRANDS)), F.array(F.lit("business"), F.lit("org_word")))
          .when(F.col("bv_any") == 1, F.array(F.lit("business"), F.lit("learned_word")))
          .when(F.col("cpty_type") == "NON_PNC_ACH_ORIGINATOR", F.array(F.lit("business"), F.lit("ach_originator")))
          .when(F.col("cpty_type") == "MERCHANT", F.array(F.lit("business"), F.lit("merchant")))
          .when(can_person & F.col("has_comma") & shape & F.col("fnc"), F.array(F.lit("person"), F.lit("comma_name")))
          .when(can_person & shape & (F.col("fn_t1") == 1), F.array(F.lit("person"), F.lit("first_name")))
          .when(can_person & shape & (F.col("fn_t2") == 1), F.array(F.lit("person"), F.lit("last_first")))
          .when(can_person & F.col("alpha") & F.col("nm_p").contains(" AND ") & (F.col("n_tok") <= 8)
                & (F.col("fn_any") == 1), F.array(F.lit("person"), F.lit("joint_names")))
          .when(wire_only & (F.col("amt") >= WIRE_INST_MIN), F.array(F.lit("business"), F.lit("wire_institution")))
          .when(~can_person, F.array(F.lit("business"), F.lit("high_value")))
          .when((F.col("cpty_type") == "P2P_CPTY") & shape, F.array(F.lit("person"), F.lit("p2p_shape")))
          .otherwise(F.array(F.lit("unknown"), F.lit("unknown"))))
    # merge key: legal and channel words removed, reference numbers removed (AMAZON MKTPL 502QZ6TY3 → AMAZON MKTPL,
    # FEDEX774833807516 → FEDEX); digit-only names keep their digits (merged with the FI in §5)
    k = F.regexp_replace(F.col("nm_c"), r"\bAT T\b", "ATT")                              # AT&T
    k = F.regexp_replace(k, r"\b(" + "|".join(KEEP_NUM_AFTER) + r") (\d+)\b", "$1_$2")     # POST 123 → POST_123 (kept)
    k = F.regexp_replace(k, r"\b([A-Z]{3,})(\d+)\b", "$1 $2")                             # PSEG973 → PSEG 973
    k = F.regexp_replace(k, r"([A-Z])(\d{4,})", "$1 $2")
    k = F.regexp_replace(k, r"(\d{3,})([A-Z])", "$1 $2")                                 # 0009THE → 0009 THE
    k = F.regexp_replace(k, r"\b(?=(?:[A-Z]*\d){4})[A-Z0-9]+\b", " ")                 # ≥ 4 digits
    k = F.regexp_replace(k, r"\b(?=[A-Z0-9]{5,}\b)(?=[A-Z]*\d[A-Z]*\d)[A-Z0-9]+\b", " ")  # long mixed codes
    k = F.regexp_replace(k, r"\b\d+\b", " ")                                           # store / reference numbers
    k = F.regexp_replace(k, wre(LEGAL + CHANNEL + ["THE"]), " ")
    k = F.trim(F.regexp_replace(k, r"\s+", " "))
    key = F.regexp_replace(k, r"(\s+[A-Z0-9])+$", "")              # VPAY E / MSFT E / INEFILE 0 → VPAY / MSFT / INEFILE
    for _pat, _canon in KEY_ALIASES.items():
        key = F.when(key.rlike(_pat), F.lit(_canon)).otherwise(key)
    v1_class = (F.when(F.coalesce(F.col("v1_ph"), F.lit(False)), "placeholder")
                 .when(F.coalesce(F.col("v1_fi"), F.lit(False)), "bank")
                 .when(F.col("name").isNull() | (F.trim(F.col("name")) == ""), "no_name")
                 .when(F.coalesce(F.col("v1_biz"), F.lit(False)), "business").otherwise("person_other"))
    out = (t.withColumn("_c", c)
            .select("cpty_id", "cpty_id_src", "ent_l1", "name", "fi", "cpty_type", "cats", "amt", "n_txn",
                    F.col("nm").alias("name_clean"),
                    F.col("_c")[0].alias("name_class"), F.col("_c")[1].alias("type_reason"),
                    v1_class.alias("v1_class"),
                    F.when(F.col("_c")[0] == "business", key).alias("biz_key"),
                    F.when(F.col("_c")[0] == "numeric_name", F.regexp_replace(F.col("nm"), r"[^0-9]", "")).alias("num_key")))
    out.write.mode("overwrite").parquet(P_TYPED)

t0 = time.time()
built = FORCE_TYPE or not path_done(P_TYPED)
if built:
    build_typed()
typed = spark.read.parquet(P_TYPED)
n_typed = typed.count()
assert n_typed == m["map_rows"], f"typed {n_typed:,} ids vs map v1 {m['map_rows']:,}: the dim and map id sets differ"

summ = (typed.groupBy("name_class", "type_reason")
             .agg(F.count("*").alias("ids"), F.sum("amt").alias("amt")).toPandas())
cls = summ.groupby("name_class")[["ids", "amt"]].sum().sort_values("amt", ascending=False).reset_index()
cls["% ids"] = 100 * cls.ids / cls.ids.sum()
cls["% $"] = 100 * cls.amt / cls.amt.sum()
cls["$B"] = cls.amt / 1e9
summ["% ids"] = 100 * summ.ids / summ.ids.sum()
summ["% $"] = 100 * summ.amt / summ.amt.sum()
summ["$B"] = summ.amt / 1e9
show(f"§3a Typer v2 — {n_typed:,} ids ({'typed' if built else 'reused'}, {(time.time() - t0) / 60:.1f} min, {path_gb(P_TYPED):.1f} GB)",
     ("By name_class (lifetime dollars)", cls[["name_class", "ids", "% ids", "$B", "% $"]]),
     ("By rule", summ.sort_values(["name_class", "amt"], ascending=[True, False])
                    [["name_class", "type_reason", "ids", "% ids", "$B", "% $"]]))

In [ ]:
# §3b  Where did the v1 classes go?
x = typed.groupBy("v1_class", "name_class").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt")).toPandas()
def xtab(metric):
    pv = x.pivot_table(index="v1_class", columns="name_class", values=metric, aggfunc="sum", fill_value=0)
    pv = 100 * pv.div(pv.sum(axis=1), axis=0)
    pv.insert(0, "v1 class", pv.index)
    return pv.reset_index(drop=True)
v1_tot = x.groupby("v1_class")[["ids", "amt"]].sum().reset_index()
v1_tot["$B"] = v1_tot.amt / 1e9
show("§3b v1 class → v2 class",
     ("v1 classes", v1_tot[["v1_class", "ids", "$B"]]),
     ("% of each v1 class's dollars, by v2 class", xtab("amt"), "pct"),
     ("% of each v1 class's ids, by v2 class", xtab("ids"), "pct"))

## §4 Eyeball check

The point of the typer is that `person` should look like people at **every** dollar level, and `unknown` should be
small in dollars. Top ids by lifetime dollars for each class, then random samples.

In [ ]:
# §4a  Largest ids by dollars in each class
EX = ["name", "name_class", "type_reason", "cpty_type", "fi", "cats", "n_txn", "amt"]
def top_of(cls_name, n=25):
    pdf = typed.where(F.col("name_class") == cls_name).orderBy(F.desc("amt")).select(*EX).limit(n).toPandas()
    return pdf.rename(columns={"amt": "$ lifetime", "n_txn": "txn", "cats": "categories"})
show("§4a Largest ids by lifetime dollars",
     ("person", top_of("person")),
     ("unknown", top_of("unknown")),
     ("business / high_value (no applicable person evidence, ≥ PERSON_MAX_AMT)",
      typed.where(F.col("type_reason") == "high_value").orderBy(F.desc("amt")).select(*EX).limit(25).toPandas()
           .rename(columns={"amt": "$ lifetime", "n_txn": "txn", "cats": "categories"})),
     ("business via behaviour or learned words (originator / merchant / wire institution / learned word)",
      typed.where(F.col("type_reason").isin("ach_originator", "merchant", "wire_institution", "learned_word"))
           .orderBy(F.desc("amt")).select(*EX).limit(25).toPandas()
           .rename(columns={"amt": "$ lifetime", "n_txn": "txn", "cats": "categories"})),
     ("numeric_name", top_of("numeric_name", 15)))

In [ ]:
# §4b  Random samples per class (seeded)
def sample_of(cls_name, n=25, seed=7):
    cnt = int(cls.loc[cls.name_class == cls_name, "ids"].sum()) if (cls.name_class == cls_name).any() else 0
    frac = min(1.0, 4.0 * n / max(cnt, 1))
    pdf = (typed.where(F.col("name_class") == cls_name).where(F.rand(seed) < frac)
                .select(*EX).limit(n).toPandas())
    return pdf.rename(columns={"amt": "$ lifetime", "n_txn": "txn", "cats": "categories"})
show("§4b Random samples",
     ("person", sample_of("person")),
     ("unknown", sample_of("unknown")),
     ("business", sample_of("business")))

## §5 Entity map v2

- `ent_l1` — from v1 (6C → 5C crosswalk).
- `ent_l2` — each `ent_l1` group takes the class and key of its target id (the 5C id).
  - Business keys merge **across FIs** (`'N:' + md5(key)`, rule `business_name`) when they are distinctive — a word of
    ≥ 4 letters used by ≤ `DF_MAX` distinct keys — **or** concentrated: ≥ `CONC_MIN` ids per FI on average (CIGNA,
    INTUIT, AMAZON MKTPL, CAPITAL ONE). National brands put many ids at each bank; local look-alikes put one or two.
  - **Generic** business keys (only common words: FIRST BAPTIST CHURCH, HOLIDAY INN EXPRESS, ORANGE COUNTY SCHOOL) merge
    only within an FI: `'F:' + md5(key | FI)`, rule `business_name_fi` — different churches at different banks stay apart.
  - Digit-only names merge on `'D:' + md5(digits | FI)`, rule `numeric_fi`.
  - Blocked keys: shorter than `KEY_MIN_LEN` (unless in `SHORT_KEYS_OK`), a single first name / organisation / legal /
    generic word, or in `NO_MERGE_KEYS` — except brand descriptors: ≥ `BRAND_BEH_SHARE` of ids are merchant or ACH
    originator and the key is concentrated (DELTA, WENDY, UNITED, KFC on card / ACH descriptors).
  - `KEY_ALIASES` folds brand spellings (AMAZON MKTPL / AMZN MKTP US / AMAZON COM → AMAZON).
  - `name_key` stays in the map, so a brand-level rollup across franchise locations is a `groupBy("name_key")` away.
    Everything else stays at `ent_l1`.
- `ent_l2_fi` — every business merge within an FI only (the strictest level).

In [ ]:
# §5a  Build and write the map
blocked_words = sorted((set(LEGAL + ORG + GENERIC_KEYS + CHANNEL + NO_MERGE_KEYS) | FIRST_NAMES) - set(SHORT_KEYS_OK))
BL_DF = F.broadcast(spark.createDataFrame(pd.DataFrame({"biz_key": blocked_words})).withColumn("blk_word", F.lit(True)))

# Distinctiveness: document frequency of each word over the distinct business keys
keys = typed.where(F.col("biz_key").isNotNull() & (F.col("biz_key") != "")).select("biz_key").distinct()
kdf = (keys.select("biz_key", F.explode(F.array_distinct(F.split("biz_key", " "))).alias("tok"))
           .where(F.col("tok").rlike(r"^[A-Z]{4,}$")))
tok_df = kdf.groupBy("tok").agg(F.count("*").alias("df"))
common = set(LEGAL + ORG + CHANNEL + GENERIC_KEYS) | FIRST_NAMES
key_rare = (kdf.join(tok_df, "tok")
               .where((F.col("df") <= DF_MAX) & ~F.col("tok").isin(sorted(common)))
               .select("biz_key").distinct().withColumn("rare", F.lit(True)))
key_conc = (typed.where(F.col("biz_key").isNotNull() & (F.col("biz_key") != ""))
                 .groupBy("biz_key")
                 .agg(F.count("*").alias("k_ids"),
                      F.countDistinct(F.coalesce(F.upper(F.trim(F.col("fi"))), F.lit("(none)"))).alias("k_fis"),
                      F.avg(F.col("type_reason").isin("merchant", "ach_originator").cast("double")).alias("beh_share")))
key_distinct = (key_conc.join(key_rare, "biz_key", "left")
                        .withColumn("conc", F.col("k_ids") >= CONC_MIN * F.col("k_fis"))
                        .withColumn("distinct_key", F.coalesce(F.col("rare"), F.lit(False)) | F.col("conc")
                                                    | F.col("biz_key").isin(list(set(KEY_ALIASES.values()))))
                        .withColumn("brand_ok", F.col("conc") & (F.col("beh_share") >= BRAND_BEH_SHARE)
                                                & (F.length("biz_key") >= 3))
                        .select("biz_key", "distinct_key", "brand_ok", "k_ids", "k_fis"))

tgt = (typed.select(F.col("cpty_id").alias("ent_l1"), F.col("name_class").alias("t_class"),
                    F.col("biz_key"), F.col("num_key").alias("t_num"), F.col("fi").alias("t_fi"))
            .join(BL_DF, "biz_key", "left")
            .join(key_distinct, "biz_key", "left")
            .withColumn("distinct_key", F.coalesce(F.col("distinct_key"), F.lit(False)))
            .withColumn("brand_ok", F.coalesce(F.col("brand_ok"), F.lit(False)) & ~F.col("biz_key").isin(NO_MERGE_KEYS))
            .withColumn("t_key_ok", F.col("biz_key").isNotNull()
                                    & ((((F.length("biz_key") >= KEY_MIN_LEN) | F.col("biz_key").isin(SHORT_KEYS_OK))
                                        & F.col("blk_word").isNull())
                                       | F.col("brand_ok"))))
fi_s = F.coalesce(F.upper(F.trim(F.col("t_fi"))), F.lit(""))
emap = (typed.select("cpty_id", "cpty_id_src", "ent_l1", "name_class", "type_reason", "name_clean")
             .join(tgt, "ent_l1", "left")
             .withColumn("ent_l0", F.col("cpty_id"))
             .withColumn("ent_l2", F.when((F.col("t_class") == "business") & F.col("t_key_ok") & F.col("distinct_key"),
                                          F.concat(F.lit("N:"), F.md5("biz_key")))
                                    .when((F.col("t_class") == "business") & F.col("t_key_ok"),
                                          F.concat(F.lit("F:"), F.md5(F.concat_ws("|", "biz_key", fi_s))))
                                    .when((F.col("t_class") == "numeric_name") & (F.length("t_num") >= 4),
                                          F.concat(F.lit("D:"), F.md5(F.concat_ws("|", "t_num", fi_s))))
                                    .otherwise(F.col("ent_l1")))
             .withColumn("ent_l2_fi", F.when(F.col("ent_l2").startswith("N:"),
                                             F.concat(F.lit("F:"), F.md5(F.concat_ws("|", "biz_key", fi_s))))
                                       .otherwise(F.col("ent_l2")))
             .withColumn("rule", F.when(F.col("ent_l2").startswith("N:"), "business_name")
                                  .when(F.col("ent_l2").startswith("F:"), "business_name_fi")
                                  .when(F.col("ent_l2").startswith("D:"), "numeric_fi")
                                  .when(F.col("ent_l1") != F.col("cpty_id"), "crosswalk_6c_5c")
                                  .otherwise("identity"))
             .withColumn("key_blocked", (F.col("t_class") == "business") & ~F.col("t_key_ok"))
             .select("cpty_id", "cpty_id_src", "ent_l0", "ent_l1", "ent_l2", "ent_l2_fi", "rule",
                     "name_class", "type_reason", "name_clean", F.col("biz_key").alias("name_key"), "key_blocked"))

if WRITE_MAP:
    t0 = time.time()
    spark.sql(f"DROP TABLE IF EXISTS {T_MAP_V2}")          # external: drops metadata only
    emap.write.mode("overwrite").parquet(P_MAP_V2)
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in spark.read.parquet(P_MAP_V2).schema.fields)
    spark.sql(f"CREATE EXTERNAL TABLE {T_MAP_V2} ({cols}) STORED AS PARQUET LOCATION '{P_MAP_V2}'")
    map_min = (time.time() - t0) / 60
else:
    map_min = float("nan")

mv2 = spark.table(T_MAP_V2) if WRITE_MAP else emap
chk = mv2.agg(F.count("*").alias("rows"), F.countDistinct("cpty_id").alias("ids"),
              F.countDistinct("ent_l1").alias("l1"), F.countDistinct("ent_l2").alias("l2"),
              F.countDistinct("ent_l2_fi").alias("l2fi")).first()
v1c = spark.table(T_MAP_V1).agg(F.countDistinct("ent_l2").alias("l2"), F.countDistinct("ent_l1").alias("l1")).first()
assert chk["rows"] == chk["ids"] == m["map_rows"], "map v2 must have one row per v1 cpty_id"
assert chk["l1"] == v1c["l1"], "ent_l1 must be unchanged from v1"

levels = pd.DataFrame([
    {"level": "raw ids (ent_l0)", "units": chk["ids"]},
    {"level": "ent_l1", "units": chk["l1"]},
    {"level": "ent_l2 — v1", "units": v1c["l2"]},
    {"level": "ent_l2 — v2", "units": chk["l2"]},
    {"level": "ent_l2_fi — v2", "units": chk["l2fi"]}])
levels["ids per unit"] = chk["ids"] / levels.units
rules = mv2.groupBy("rule").agg(F.count("*").alias("ids")).toPandas().sort_values("ids", ascending=False)
show(f"§5a Entity map v2 → {T_MAP_V2} ({map_min:.1f} min)",
     ("Units per level", levels), ("Ids per rule", rules),
     note(f"Impala: INVALIDATE METADATA {T_MAP_V2};"))

In [ ]:
# §5b  Merge evidence: largest merges, blocked keys, numeric merges
dims = typed.select("cpty_id", "amt", "fi", "name")
mm = mv2.join(dims, "cpty_id")
big = (mm.where(F.col("rule") == "business_name").groupBy("ent_l2")
         .agg(F.first("name_key").alias("key"), F.count("*").alias("ids"), F.countDistinct("fi").alias("FIs"),
              F.countDistinct("ent_l2_fi").alias("name+FI units"), F.sum("amt").alias("amt"))
         .orderBy(F.desc("ids")).limit(30).toPandas())
big["$M lifetime"] = big.amt / 1e6
gen = (mm.where(F.col("rule") == "business_name_fi").groupBy("name_key")
         .agg(F.count("*").alias("ids"), F.countDistinct("fi").alias("FIs"),
              F.countDistinct("ent_l2").alias("units (within FI)"), F.sum("amt").alias("amt"))
         .orderBy(F.desc("ids")).limit(25).toPandas())
gen["$M lifetime"] = gen.amt / 1e6
gen["ids per FI"] = gen.ids / gen.FIs.clip(lower=1)
empty = (mm.where(F.col("key_blocked") & (F.coalesce(F.col("name_key"), F.lit("")) == ""))
           .orderBy(F.desc("amt")).select("name", "fi", "amt").limit(15).toPandas())
empty["$M lifetime"] = empty.amt / 1e6
scope = (mm.groupBy("rule").agg(F.count("*").alias("ids"), F.countDistinct("ent_l2").alias("units"),
                                F.sum("amt").alias("amt")).toPandas().sort_values("ids", ascending=False))
scope["$B lifetime"] = scope.amt / 1e9
num = (mm.where(F.col("rule") == "numeric_fi").groupBy("ent_l2")
         .agg(F.first("name_clean").alias("name"), F.first("fi").alias("fi"), F.count("*").alias("ids"),
              F.sum("amt").alias("amt"))
         .orderBy(F.desc("amt")).limit(15).toPandas())
num["$M lifetime"] = num.amt / 1e6
blk = (mm.where(F.col("key_blocked")).groupBy(F.coalesce("name_key", F.lit("(empty)")).alias("key"))
         .agg(F.count("*").alias("ids"), F.sum("amt").alias("amt")).orderBy(F.desc("ids")).limit(20).toPandas())
blk["$M lifetime"] = blk.amt / 1e6
show("§5b Merge evidence",
     ("Ids, units and dollars per rule", scope[["rule", "ids", "units", "$B lifetime"]]),
     ("Largest cross-FI merges (distinctive keys)", big[["key", "ids", "FIs", "name+FI units", "$M lifetime"]]),
     (f"Largest generic keys (merged only within an FI; DF_MAX = {DF_MAX})",
      gen[["name_key", "ids", "FIs", "ids per FI", "units (within FI)", "$M lifetime"]].rename(columns={"name_key": "key"})),
     ("Largest numeric-name merges (digits + FI)", num[["name", "fi", "ids", "$M lifetime"]]),
     ("Most frequent blocked keys (business ids left unmerged)", blk[["key", "ids", "$M lifetime"]]),
     ("Largest business ids whose key is empty after cleaning", empty[["name", "fi", "$M lifetime"]]))

## §6 Next

Rerun `pkg_cpty_threshold_explore.ipynb` with `T_MAP = f"{DB}.pkg_cpty_entity_map_v2"`. It reads `name_class` from the
map, caches its intermediates under a separate folder, and its candidate rules can then be restricted to classes, e.g.
`{"amt": 1_000, "classes": BIZ_LIKE}`.